In [31]:
import numpy as np
import pandas as pd
import re

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

In [32]:
train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")

In [33]:
y = train_df["Survived"].astype(int)
train_X = train_df.drop(columns=["Survived"])

In [34]:
full = pd.concat([train_X, test_df], axis=0, ignore_index=True)
full

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
1304,1305,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
1305,1306,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
1306,1307,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
1307,1308,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [35]:
full["Title"] = full["Name"].str.extract(r" ([A-Za-z]+)\.", expand=True)
title_map = {
    "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
    "Major": "Mr", "Col": "Mr", "Sir": "Mr", "Don": "Mr", "Jonkheer": "Mr", "Capt": "Mr",
    "Lady": "Mrs", "Countess": "Mrs", "Dona": "Mrs"
}

In [36]:
full["Title"] = full["Title"].replace(title_map)
# Chỉ giữ các nhóm chính; cái khác xem như "Other" để median ổn định
major_titles = {"Mr","Mrs","Miss","Master","Dr","Rev"}
full["Title"] = np.where(full["Title"].isin(major_titles), full["Title"], "Other")

In [37]:
age_median_by_title = full.groupby("Title")["Age"].median()
full["Age"] = full["Age"].fillna(full["Title"].map(age_median_by_title))
# fallback nếu còn NaN
full["Age"] = full["Age"].fillna(full["Age"].median())

In [38]:
full["Family_Size"] = full["Parch"].fillna(0) + full["SibSp"].fillna(0)

In [39]:
tmp = train_df.copy()
tmp["Last_Name"] = tmp["Name"].str.split(",").str[0]
tmp["Fare"] = tmp["Fare"].fillna(tmp["Fare"].median())


In [40]:
DEFAULT_SURV = 0.5
fam_rules = {}

for (lname, fare), grp in tmp.groupby(["Last_Name", "Fare"], dropna=False):
    if len(grp) <= 1:
        continue
    smax = grp["Survived"].max()
    smin = grp["Survived"].min()
    if smax == 1:
        fam_rules[("LF", lname, fare)] = 1.0
    elif smin == 0:
        fam_rules[("LF", lname, fare)] = 0.0

In [41]:
# Quy tắc theo Ticket trong TRAIN
for ticket, grp in tmp.groupby("Ticket", dropna=False):
    if len(grp) <= 1:
        continue
    smax = grp["Survived"].max()
    smin = grp["Survived"].min()
    if smax == 1:
        fam_rules[("T", ticket)] = 1.0
    elif smin == 0:
        fam_rules[("T", ticket)] = 0.0

In [42]:
full["Last_Name"] = full["Name"].str.split(",").str[0]
full["Fare"] = full["Fare"].astype(float)
full["Fare"] = full.groupby("Pclass")["Fare"].transform(lambda s: s.fillna(s.median()))

In [43]:
def infer_family_survival(row):
    # Ưu tiên quy tắc theo Last_Name + Fare
    key1 = ("LF", row["Last_Name"], row["Fare"])
    if key1 in fam_rules:
        return fam_rules[key1]
    # Sau đó đến Ticket
    key2 = ("T", row["Ticket"])
    if key2 in fam_rules:
        return fam_rules[key2]
    return DEFAULT_SURV

full["Family_Survival"] = full.apply(infer_family_survival, axis=1)

In [44]:
# FareBin / AgeBin (ordinal) → mã hoá bằng cat.codes
full["Fare"] = full["Fare"].fillna(full["Fare"].median())
full["FareBin"] = pd.qcut(full["Fare"], 5, duplicates="drop")
full["FareBin_Code"] = full["FareBin"].cat.codes



full["AgeBin"] = pd.qcut(full["Age"], 4, duplicates="drop")
full["AgeBin_Code"] = full["AgeBin"].cat.codes


In [45]:
full["Sex"] = full["Sex"].replace({"male":0, "female":1})

/tmp/ipykernel_28566/4221836729.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  full["Sex"] = full["Sex"].replace({"male":0, "female":1})


In [46]:
drop_cols = ["Name","PassengerId","SibSp","Parch","Ticket","Cabin","Embarked","Fare","Age","FareBin","AgeBin","Last_Name"]
X_full = full.drop(columns=[c for c in drop_cols if c in full.columns])

In [47]:
title_order = {"Mr":0, "Mrs":1, "Miss":2, "Master":3, "Dr":4, "Rev":5, "Other":6}
X_full["Title"] = X_full["Title"].map(title_order).astype(int)


In [48]:
X = X_full.iloc[:len(train_df)].reset_index(drop=True)
X_test = X_full.iloc[len(train_df):].reset_index(drop=True)


In [51]:
# ========= 4) SCALING =========
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

In [52]:
param_grid = {
    "n_neighbors": [6,7,8,9,10,11,12,14,16,18,20,22],
    "weights": ["uniform", "distance"],
    "leaf_size": list(range(1, 50, 5)),
    "algorithm": ["auto"],   # có thể thêm 'ball_tree','kd_tree' nếu muốn thử
    "p": [1, 2],             # Manhattan / Euclidean
}

In [ ]:
knn = KNeighborsClassifier()
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1
)
grid.fit(X_scaled, y)

print("Best ROC_AUC:", grid.best_score_)
print("Best Estimator:", grid.best_estimator_)

Fitting 10 folds for each of 480 candidates, totalling 4800 fits
